# MSA Preparation



In [11]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
from pathlib import Path

In [12]:
# Alphabet: 20 amino acids + gap
ALPHABET = "ARNDCQEGHILKMFPSTWYV-"
A2I = {aa: i for i, aa in enumerate(ALPHABET)}
I2A = {i: aa for i, aa in enumerate(ALPHABET)}
A = len(ALPHABET)  # 21

## Functions

In [ ]:
def parse_a3m(path):
    """Parse a3m file to headers and sequences.
    
    a3m format: lowercase = insertions (ignored), uppercase = aligned positions.
    """
    headers, sequences = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                headers.append(line[1:])
                sequences.append([])
            elif line and sequences:
                # Keep only uppercase (aligned) and gaps
                aligned = "".join(c for c in line if c.isupper() or c == "-")
                sequences[-1].append(aligned)
    sequences = ["".join(s) for s in sequences]
    return headers, sequences

In [ ]:
def make_msa(sequences, gap_thresh=0.5, eff_thresh=0.8):
    """Convert sequences to MSA tensors.
    
    Args:
        sequences: list of aligned sequences (same length)
        gap_thresh: remove columns with gap frequency above this
        eff_thresh: cluster sequences at this identity for weighting
    
    Returns:
        dict with:
            X: one-hot MSA (N, L, A) - filtered columns, 20 AAs (no gap)
            X_raw: one-hot MSA (N, L_raw, A+1) - all columns, 21 chars
            non_gap: indices of non-gapped columns
            weights: effective sequence weights (N,)
    """
    # Sequences to integer indices
    raw_idx = np.array([
        [A2I.get(aa.upper(), A2I["-"]) for aa in seq] 
        for seq in sequences
    ])
    N, L_raw = raw_idx.shape
    # Find non-gapped columns
    gap_freq = (raw_idx == A2I["-"]).mean(axis=0)
    non_gap = np.where(gap_freq < gap_thresh)[0]
    L = len(non_gap)
    # Compute sequence weights (effective sequences)
    identity = 1.0 - squareform(pdist(raw_idx, "hamming"))
    weights = 1.0 / (identity >= eff_thresh).sum(axis=1)
    # One-hot encode
    X_raw = np.eye(A)[raw_idx]  # (N, L_raw, 21) with gap
    X = np.eye(A)[raw_idx[:, non_gap]][:, :, :(A - 1)]  # (N, L, 20) no gap
    return {
        "X": X.astype(np.float32),
        "X_raw": X_raw.astype(np.float32),
        "non_gap": non_gap,
        "weights": weights.astype(np.float32),
    }

## Data Collation

source: https://alphafold.ebi.ac.uk/entry/AF-P0AA25-F1

In [15]:
MSA_PATH = "AF-P0AA25-F1-msa_v6.a3m"
headers, sequences = parse_a3m(MSA_PATH)
print(f"Parsed {len(sequences)} sequences")
print(f"Raw length: {len(sequences[0])}")

Parsed 19413 sequences
Raw length: 109


In [16]:
msa = make_msa(sequences)

In [17]:
N, L, A = msa["X"].shape
N_eff = msa["weights"].sum()

print(f"X shape: {msa['X'].shape}")
print(f"X_raw shape: {msa['X_raw'].shape}")
print(f"Effective sequences: {N_eff:.1f}")
print(f"Non-gap columns: {len(msa['non_gap'])} / {msa['X_raw'].shape[1]}")

X shape: (19413, 101, 20)
X_raw shape: (19413, 109, 21)
Effective sequences: 9378.7
Non-gap columns: 101 / 109


In [18]:
for x, y in msa.items():
    print(f"{x}: {y.shape if isinstance(y, np.ndarray) else y}")

X: (19413, 101, 20)
X_raw: (19413, 109, 21)
non_gap: (101,)
weights: (19413,)


## Save

In [19]:
out_path = Path(MSA_PATH).stem + ".npz"
np.savez(out_path, **msa)
print(f"Saved to {out_path}")

Saved to AF-P0AA25-F1-msa_v6.npz
